# 22.3 — Informed search & A*

A* expands the state with the smallest cost-so-far plus a safe estimate of cost-to-go.

UCS respects path cost; A* adds a heuristic estimate of what remains. The frontier becomes smaller when the heuristic is informative and still safe when it is admissible.

Save a copy to Drive to edit.

In [ ]:
import heapq
import math
import random
import matplotlib.pyplot as plt

SEED = 223
random.seed(SEED)


def manhattan(goal):
    def h(node):
        return abs(goal[0] - node[0]) + abs(goal[1] - node[1])
    return h


def grid_neighbors(width, height, walls=None):
    walls = set(walls or [])

    def neighbors(node):
        row, col = node
        out = []
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nxt = (row + dr, col + dc)
            if 0 <= nxt[0] < height and 0 <= nxt[1] < width and nxt not in walls:
                out.append(nxt)
        return out

    return neighbors


def terrain_cost(expensive=None, penalty=5):
    expensive = set(expensive or [])

    def cost(current, nxt):
        if nxt in expensive:
            return penalty
        return 1

    return cost


def make_astar_instances():
    instances = []
    instances.append({"name": "D1 5x5 trace", "width": 5, "height": 5, "start": (0, 0), "goal": (4, 4), "walls": set(), "expensive": set(), "penalty": 1})
    instances.append({"name": "D2 wider open grid", "width": 9, "height": 7, "start": (0, 0), "goal": (6, 8), "walls": set(), "expensive": set(), "penalty": 1})
    instances.append({"name": "D3 maze with walls", "width": 10, "height": 8, "start": (0, 0), "goal": (7, 9), "walls": {(1, 2), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4), (2, 6), (3, 6), (4, 6), (5, 6)}, "expensive": set(), "penalty": 1})
    instances.append({"name": "D4 weighted terrain", "width": 12, "height": 9, "start": (0, 0), "goal": (8, 11), "walls": {(2, 4), (3, 4), (4, 4), (5, 4), (6, 4)}, "expensive": {(4, c) for c in range(6, 11)}, "penalty": 8})
    instances.append({"name": "D5 deceptive overestimate", "width": 14, "height": 10, "start": (0, 0), "goal": (9, 13), "walls": {(r, 5) for r in range(0, 8)} | {(r, 9) for r in range(2, 10)}, "expensive": {(1, c) for c in range(1, 13)} | {(r, 13) for r in range(2, 9)}, "penalty": 10})
    return instances


## The concept, built once (D1)

A* uses $$f(n)=g(n)+h(n)$$ where $g(n)$ is exact cost so far and $h(n)$ estimates cost-to-go. With $h=0$, A* becomes UCS; with an overestimate, optimality can fail.

In [ ]:
def rebuild(parent, node):
    path = [node]
    while parent[node] is not None:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path


def astar(neighbors, start, goal, h, cost_fn=None, weight=1):
    cost_fn = cost_fn or (lambda current, nxt: 1)
    frontier = [(weight * h(start), 0, start)]
    parent = {start: None}
    g_score = {start: 0}
    expanded = []
    counter = 1

    while frontier:
        _, _, node = heapq.heappop(frontier)
        if node in expanded:
            continue
        expanded.append(node)
        if node == goal:
            return {
                "path": rebuild(parent, node),
                "cost": g_score[node],
                "expanded": expanded,
            }
        for nxt in neighbors(node):
            tentative = g_score[node] + cost_fn(node, nxt)
            if tentative < g_score.get(nxt, math.inf):
                parent[nxt] = node
                g_score[nxt] = tentative
                priority = tentative + weight * h(nxt)
                heapq.heappush(frontier, (priority, counter, nxt))
                counter = counter + 1

    return {
        "path": [],
        "cost": math.inf,
        "expanded": expanded,
    }


The lesson's exact checks are Manhattan $|4|+|4|=8$, goal heuristic $0$, path cost $9-1=8$, weighted-A* start priority $0+2\cdot8=16$, and a node with $g=3,h=5$ having priority $13$.

In [ ]:
d1_neighbors = grid_neighbors(5, 5)
d1_h = manhattan((4, 4))
d1_result = astar(d1_neighbors, (0, 0), (4, 4), d1_h)
start_priority = 0 + 2 * d1_h((0, 0))
node_priority = 3 + 2 * 5

assert d1_h((0, 0)) == 8
assert d1_h((4, 4)) == 0
assert d1_result["cost"] == 8
assert len(d1_result["path"]) - 1 == 8
assert start_priority == 16
assert node_priority == 13


## The dataset ladder

The ladder grows from a tiny trace to open grids, wall mazes, weighted terrain, and D5 with an overestimating deceptive heuristic.

In [ ]:
instances = make_astar_instances()
for item in instances:
    print(item["name"], f"{item['height']}x{item['width']}", "walls", len(item["walls"]), "expensive", len(item["expensive"]))


In [ ]:
rows = []
for item in instances:
    neighbors = grid_neighbors(item["width"], item["height"], item["walls"])
    costs = terrain_cost(item["expensive"], item["penalty"])
    h_safe = manhattan(item["goal"])
    optimal = astar(neighbors, item["start"], item["goal"], lambda node: 0, costs)
    informed = astar(neighbors, item["start"], item["goal"], h_safe, costs)
    rows.append({
        "rung": item["name"].split()[0],
        "optimal_cost": optimal["cost"],
        "astar_cost": informed["cost"],
        "optimality_gap": informed["cost"] - optimal["cost"],
        "ucs_expanded": len(optimal["expanded"]),
        "astar_expanded": len(informed["expanded"]),
        "path": informed["path"],
        "expanded": informed["expanded"],
    })

for row in rows:
    print(row)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, item, row in zip(axes[0], instances, rows):
    ax.set_title(item["name"].split()[0])
    expanded = set(row["expanded"])
    path = set(row["path"])
    for r in range(item["height"]):
        for c in range(item["width"]):
            color = "white"
            if (r, c) in item["walls"]:
                color = "black"
            elif (r, c) in item["expensive"]:
                color = "orange"
            elif (r, c) in expanded:
                color = "lightblue"
            if (r, c) in path:
                color = "lime"
            ax.add_patch(plt.Rectangle((c, item["height"] - 1 - r), 1, 1, facecolor=color, edgecolor="gray"))
    ax.set_xlim(0, item["width"])
    ax.set_ylim(0, item["height"])
    ax.axis("off")

labels = [row["rung"] for row in rows]
axes[1, 0].plot(labels, [row["optimality_gap"] for row in rows], marker="o")
axes[1, 0].set_title("optimality gap")
axes[1, 1].plot(labels, [row["astar_expanded"] for row in rows], marker="o", label="A*")
axes[1, 1].plot(labels, [row["ucs_expanded"] for row in rows], marker="o", label="UCS")
axes[1, 1].set_title("nodes expanded")
axes[1, 1].legend()
for ax in axes[1, 2:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: overestimating with $h$

A heuristic that over-penalizes the cheaper detour can make the true optimum look too expensive. The fix is to restore admissible Manhattan or use the UCS fallback $h=0$.

In [ ]:
d5 = instances[-1]
d5_neighbors = grid_neighbors(d5["width"], d5["height"], d5["walls"])
d5_cost = terrain_cost(d5["expensive"], d5["penalty"])

def deceptive_h(node):
    row, col = node
    penalty = 25 if row >= 8 and col < 10 else 0
    return manhattan(d5["goal"])(node) + penalty

wrong = astar(d5_neighbors, d5["start"], d5["goal"], deceptive_h, d5_cost)
fixed = astar(d5_neighbors, d5["start"], d5["goal"], manhattan(d5["goal"]), d5_cost)
fallback = astar(d5_neighbors, d5["start"], d5["goal"], lambda node: 0, d5_cost)
print("deceptive cost", wrong["cost"])
print("admissible cost", fixed["cost"])
print("UCS cost", fallback["cost"])


## Evaluate it + Practice

- Main metric: optimality gap and nodes expanded.
- No-skill baseline: $h=0$ UCS.
- Cheap sanity check: rerun D1 and verify the hand-trace number from the lesson exactly.
- Ablation: multiply Manhattan by 2 and watch the proof disappear on D5.
- Failure signals: negative gaps, a heuristic with different units than cost, or fewer nodes paired with a worse path without reporting the gap.

Practice prompts:
1. Change one D3 obstacle, edge, or score parameter and predict which nodes change before running.
2. Add one more D5 deceptive branch and compare the metric table.
3. Write a one-sentence rule for when the pitfall would be dangerous in production.